# 03 training and optuna

In [ ]:
if "config" not in globals():
    raise RuntimeError("Run 00_experiment_runner.ipynb first so config is defined.")
trained_models = {}
training_histories = {}
best_params_by_model = {}
results = {}
training_summary = {}


## 학습 설정

In [ ]:
if "config" not in globals():
    raise RuntimeError("Run 00_experiment_runner.ipynb first so config is defined.")

trained_models = {}
training_histories = {}
best_params_by_model = {}
results = {}
training_summary = {}


In [ ]:
def run_train_and_validate(config, trial=None, return_history=False):
    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config["batch_size"],
        shuffle=False,
    )

    model_name = config.get("model_name", "LSTM")
    model = build_model(config).to(device)
    print(
        f"현재 학습 모델: {model_name} | loss: {config.get('loss_type', 'mse')} "
        f"| output: (B, {config['model_output_len']}, {config['target_dim']})"
    )
    print("model device:", next(model.parameters()).device)

    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])

    best_v_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    best_epoch = 0
    counter = 0

    train_losses = []
    val_losses = []
    last_epoch = 0
    last_v_loss = float("inf")

    heave_idx = int(config["heave_idx"])
    heave_loss_weight = float(config.get("heave_loss_weight", 0.0))

    def calculate_total_loss(pred, target):
        if pred.shape != target.shape:
            raise RuntimeError(
                f"prediction/target shape 불일치: {pred.shape} vs {target.shape}"
            )

        base_loss = compute_loss(pred, target, config)
        heave_aux_loss = F.mse_loss(
            pred[..., heave_idx],
            target[..., heave_idx],
        )
        return base_loss + heave_loss_weight * heave_aux_loss

    for epoch in range(config["epoch"]):
        model.train()
        train_loss_sum = 0.0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)  # (B, 100, 2)

            optimizer.zero_grad()
            pred = model(xb)    # (B, 100, 2)
            loss = calculate_total_loss(pred, yb)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                config["grad_clip"],
            )
            optimizer.step()
            train_loss_sum += loss.item()

        avg_train_loss = train_loss_sum / len(train_loader)
        train_losses.append(float(avg_train_loss))

        model.eval()
        val_loss_sum = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                pred = model(xb)
                val_loss_sum += calculate_total_loss(pred, yb).item()

        avg_val_loss = val_loss_sum / len(val_loader)
        val_losses.append(float(avg_val_loss))
        last_epoch = epoch + 1
        last_v_loss = float(avg_val_loss)

        print(
            f"[Epoch {epoch + 1:03d}] "
            f"Train: {avg_train_loss:.6f} | Val: {avg_val_loss:.6f}"
        )

        if trial is not None:
            trial.report(avg_val_loss, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        if avg_val_loss < best_v_loss:
            best_v_loss = float(avg_val_loss)
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            counter = 0
        else:
            counter += 1

        if counter >= config["patience"]:
            print(
                f"Early stopping at epoch {epoch + 1} "
                f"| best epoch: {best_epoch}"
            )
            break

    checkpoint_paths = {}
    should_save_ckpt = (
        config.get("save_checkpoints", False)
        and trial is None
        and return_history
    )

    if should_save_ckpt:
        checkpoint_paths["last"] = save_checkpoint(
            model=model,
            config=config,
            model_name=model_name,
            tag="last",
            epoch=last_epoch,
            val_loss=last_v_loss,
        )

    model.load_state_dict(best_state)
    model.eval()

    if should_save_ckpt:
        checkpoint_paths["best"] = save_checkpoint(
            model=model,
            config=config,
            model_name=model_name,
            tag="best",
            epoch=best_epoch,
            val_loss=best_v_loss,
        )
        print("checkpoint 저장:", checkpoint_paths)

    if return_history:
        return (
            best_v_loss,
            train_losses,
            val_losses,
            model,
            best_epoch,
            checkpoint_paths,
        )
    return best_v_loss


## Optuna

In [ ]:
def suggest_params_by_model(trial, model_name):
    if model_name in ["RNN", "LSTM", "GRU", "BiLSTM"]:
        hidden_size = trial.suggest_categorical(
            "hidden_size",
            [16, 32, 64, 128, 256]
        )

        num_layers = trial.suggest_int(
            "num_layers",
            1,
            4
        )

        if num_layers == 1:
            dropout = 0.0
        else:
            dropout = trial.suggest_float(
                "dropout",
                0.0,
                0.35
            )

        params = {
            "hidden_size": hidden_size,
            "num_layers": num_layers,
            "dropout": dropout,
            "lr": trial.suggest_float("lr", 1e-4, 5e-3, log=True),
            "grad_clip": trial.suggest_categorical("grad_clip", [0.5, 1.0, 2.0, 5.0]),
        }

        return params

    elif model_name == "TCN":
        num_levels = trial.suggest_int("tcn_num_levels", 2, 4)
        base_channels = trial.suggest_categorical("tcn_base_channels", [16, 32, 64])

        channels = [base_channels * (2 ** min(i, 2)) for i in range(num_levels)]

        params = {
            "tcn_channels": channels,
            "tcn_kernel_size": trial.suggest_categorical("tcn_kernel_size", [2, 3, 5]),
            "tcn_dropout": trial.suggest_float("tcn_dropout", 0.0, 0.3),
            "lr": trial.suggest_float("lr", 1e-4, 5e-3, log=True),
            "grad_clip": trial.suggest_categorical("grad_clip", [0.5, 1.0, 2.0, 5.0]),
        }

        return params
    elif model_name == "Transformer":
        hidden_size = trial.suggest_categorical(
            "hidden_size",
            [32, 64, 128, 256]
        )

        possible_heads = [
            h for h in [1, 2, 4, 8]
            if hidden_size % h == 0
        ]

        nhead = trial.suggest_categorical(
            "nhead",
            possible_heads
        )

        num_layers = trial.suggest_int("num_layers",2,5)
        params = {
            "hidden_size": hidden_size,
            "num_layers": num_layers,
            "nhead": nhead,
            "dim_feedforward": trial.suggest_categorical(
                "dim_feedforward",
                [128, 256, 512, 1024]
            ),
            "dropout": trial.suggest_float("dropout", 0.05, 0.35),
            "lr": trial.suggest_float("lr", 5e-5, 3e-3, log=True),
            "grad_clip": trial.suggest_categorical("grad_clip", [0.5, 1.0, 2.0, 5.0]),
        }

        return params


        return params

    else:
        raise ValueError(f"지원하지 않는 model_name: {model_name}")

## 최종 학습 및 체크포인트

In [ ]:
import os
import json
import optuna

def load_optuna_params_by_model(config):
    path = config["optuna_best_params_path"]
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_optuna_params_by_model(config, params_by_model):
    os.makedirs(get_experiment_dir(config), exist_ok=True)
    path = config["optuna_best_params_path"]
    with open(path, "w", encoding="utf-8") as f:
        json.dump(params_by_model, f, indent=4, ensure_ascii=False)
    print("Optuna params 저장:", path)


def run_or_load_optuna_for_model(config, model_name, saved_params_by_model):
    """
    각 모델별로 Optuna를 실행하고 best params를 저장합니다.
    이미 해당 모델의 best params가 있으면 재사용합니다.
    """
    if model_name in saved_params_by_model:
        print(f"✅ 기존 Optuna best params 로드: {model_name}")
        print(saved_params_by_model[model_name])
        return saved_params_by_model[model_name]["best_params"]

    if not config.get("run_optuna", True):
        print(f"⚠️ run_optuna=False이고 {model_name} 저장 params가 없음 → 기본 config 사용")
        return {}

    print("=" * 70)
    print(f"🔎 Optuna 시작: {model_name}")
    print("=" * 70)

    def objective(trial):
        trial_config = config.copy()

        suggested_params = suggest_params_by_model(trial, model_name)

        trial_config.update({
            "model_name": model_name,
            "epoch": config.get("optuna_epoch", 50),
            "patience": config.get("optuna_patience", 7),
            "input_dim": config["input_dim"],
            "target_dim": config["target_dim"],
            "input_len": config["input_len"],
            "model_output_len": config["model_output_len"],
            "eval_output_len": config["eval_output_len"],
            "stride": config["stride"],
        })

        trial_config.update(suggested_params)

        return run_train_and_validate(
            trial_config,
            trial=trial,
            return_history=False
        )

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=config.get("optuna_n_trials", 30))

    record = {
        "model_name": model_name,
        "best_value": float(study.best_value),
        "best_params": study.best_params,
        "n_trials": len(study.trials),
        "model_output_len": config["model_output_len"],
        "eval_output_len": config["eval_output_len"],
        "experiment_name": config.get("experiment_name", "default_experiment"),
    }

    saved_params_by_model[model_name] = record
    save_optuna_params_by_model(config, saved_params_by_model)

    print(f"✅ {model_name} Optuna 완료")
    print(record)
    return record["best_params"]


model_names = list(config["model_names"])
best_params_by_model = {}
if config["run_optuna"]:
    optuna_params_by_model = {}
    for model_name in model_names: best_params_by_model[model_name] = run_or_load_optuna_for_model(config, model_name, optuna_params_by_model)
elif config["use_optuna_best_params"]:
    if not os.path.exists(config["optuna_best_params_path"]): raise FileNotFoundError(f"Saved Optuna parameters not found: {config['optuna_best_params_path']}")
    optuna_params_by_model = load_optuna_params_by_model(config)
    for model_name in model_names:
        if model_name not in optuna_params_by_model: raise FileNotFoundError(f"Saved Optuna parameters missing for {model_name}")
        best_params_by_model[model_name] = optuna_params_by_model[model_name]["best_params"]
else:
    print("Optuna disabled: training with config defaults.")
    best_params_by_model = {name: {} for name in model_names}

results = {}
training_summary = {}

for model_name in model_names:
    print("=" * 70)
    print(f"🚀 최종 학습 시작: {model_name}")
    print("=" * 70)

    final_config = config.copy()

    final_config.update({
        "model_name": model_name,
        "input_len": config["input_len"],
        "model_output_len": config["model_output_len"],
        "eval_output_len": config["eval_output_len"],
        "stride": config["stride"],
        "input_dim": config["input_dim"],
            "target_dim": config["target_dim"],
        "epoch": config["epoch"],
        "patience": config["patience"],
        "results_dir": config["results_dir"],
        "experiment_name": config["experiment_name"],
        "experiment_dir": config["experiment_dir"],
        "checkpoint_dir": config["checkpoint_dir"],
        "loss_type": config.get("loss_type", "mse"),
        "t_peak": config.get("t_peak", None),
    })

    final_config.update(best_params_by_model.get(model_name, {}))

    if model_name == "Transformer" and final_config["hidden_size"] % final_config["nhead"] != 0:
        valid_heads = [h for h in [1, 2, 4, 8] if final_config["hidden_size"] % h == 0]
        final_config["nhead"] = valid_heads[0]
        print(f"Transformer nhead 자동 보정: {final_config['nhead']}")

    best_v_loss, train_losses, val_losses, model, best_epoch, checkpoint_paths = run_train_and_validate(
        final_config,
        trial=None,
        return_history=True
    )

    results[model_name] = {
        "best_v_loss": best_v_loss,
        "best_epoch": best_epoch,
        "train_losses": train_losses,
        "val_losses": val_losses,
        "model": model, 
        "config": final_config,
        "checkpoint_paths": checkpoint_paths,
    }

    training_summary[model_name] = {
        "best_v_loss": float(best_v_loss),
        "best_epoch": int(best_epoch),
        "best_params": best_params_by_model.get(model_name, {}),
        "loss_type": final_config.get("loss_type", "mse"),
        "checkpoint_paths": checkpoint_paths,
        "train_losses": train_losses,
        "val_losses": val_losses,
        "config": {k: v for k, v in final_config.items() if k != "model"},
    }

    print(f"✅ {model_name} 완료 | Best Val Loss: {best_v_loss:.6f} | Best Epoch: {best_epoch}")

summary_path = os.path.join(get_experiment_dir(config), "training_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(training_summary, f, indent=4, ensure_ascii=False)

print("=" * 70)
print("전체 모델 학습 완료")
print("summary 저장:", summary_path)
print("Optuna params 저장:", config["optuna_best_params_path"])
print("checkpoint dir:", config.get("checkpoint_dir"))
print("=" * 70)